In [6]:
import pandas as pd

df = pd.read_csv("primary_dataset.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nResult values:")
print(df["result"].value_counts())

print("\nUnique result values:")
print(df["result"].unique())


Shape: (16411, 7)

Columns:
Index(['crop ID', 'soil_type', 'Seedling Stage', 'MOI', 'temp', 'humidity',
       'result'],
      dtype='object')

Result values:
result
0    9062
1    6227
2    1122
Name: count, dtype: int64

Unique result values:
[1 2 0]


In [7]:
import pandas as pd

df = pd.read_csv("primary_dataset.csv")

print(df.groupby("result")[["MOI","temp","humidity"]].mean())

              MOI       temp   humidity
result                                 
0       48.275215  23.729089  74.453542
1       32.717199  35.074996  50.073936
2       67.633690  35.407308  49.366399


In [8]:
import pandas as pd

df = pd.read_csv("primary_dataset.csv")

print("crop ID:")
print(df["crop ID"].unique())

print("\nsoil_type:")
print(df["soil_type"].unique())

print("\nSeedling Stage:")
print(df["Seedling Stage"].unique())

crop ID:
['Wheat' 'Potato' 'Carrot' 'Tomato' 'Chilli']

soil_type:
['Black Soil' 'Alluvial Soil' 'Sandy Soil' 'Red Soil' 'Clay Soil'
 'Loam Soil' 'Chalky Soil']

Seedling Stage:
['Germination' 'Seedling Stage'
 'Vegetative Growth / Root or Tuber Development' 'Flowering' 'Pollination'
 'Fruit/Grain/Bulb Formation' 'Maturation' 'Harvest']


In [9]:
import pandas as pd

df = pd.read_csv("primary_dataset.csv")

print("Missing Values:")
print(df.isnull().sum())

print("\nCrop ID Values:")
print(df["crop ID"].unique())

print("\nSoil Type Values:")
print(df["soil_type"].unique())

print("\nSeedling Stage Values:")
print(df["Seedling Stage"].unique())


Missing Values:
crop ID           0
soil_type         0
Seedling Stage    0
MOI               0
temp              0
humidity          0
result            0
dtype: int64

Crop ID Values:
['Wheat' 'Potato' 'Carrot' 'Tomato' 'Chilli']

Soil Type Values:
['Black Soil' 'Alluvial Soil' 'Sandy Soil' 'Red Soil' 'Clay Soil'
 'Loam Soil' 'Chalky Soil']

Seedling Stage Values:
['Germination' 'Seedling Stage'
 'Vegetative Growth / Root or Tuber Development' 'Flowering' 'Pollination'
 'Fruit/Grain/Bulb Formation' 'Maturation' 'Harvest']


In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("primary_dataset.csv")

crop_encoder = LabelEncoder()
soil_encoder = LabelEncoder()
stage_encoder = LabelEncoder()

df["crop_ID_encoded"] = crop_encoder.fit_transform(df["crop ID"])
df["soil_type_encoded"] = soil_encoder.fit_transform(df["soil_type"])
df["stage_encoded"] = stage_encoder.fit_transform(df["Seedling Stage"])

df.head()

,crop ID,soil_type,Seedling Stage,MOI,temp,humidity,result,crop_ID_encoded,soil_type_encoded,stage_encoded
0,Wheat,Black Soil,Germination,1,25,80.0,1,4,1,2
1,Wheat,Black Soil,Germination,2,26,77.0,1,4,1,2
2,Wheat,Black Soil,Germination,3,27,74.0,1,4,1,2
3,Wheat,Black Soil,Germination,4,28,71.0,1,4,1,2
4,Wheat,Black Soil,Germination,5,29,68.0,1,4,1,2


In [11]:
print("Crop Mapping:")
for i, label in enumerate(crop_encoder.classes_):
    print(i, "->", label)

print("\nSoil Mapping:")
for i, label in enumerate(soil_encoder.classes_):
    print(i, "->", label)

print("\nStage Mapping:")
for i, label in enumerate(stage_encoder.classes_):
    print(i, "->", label)

Crop Mapping:
0 -> Carrot
1 -> Chilli
2 -> Potato
3 -> Tomato
4 -> Wheat

Soil Mapping:
0 -> Alluvial Soil
1 -> Black Soil
2 -> Chalky Soil
3 -> Clay Soil
4 -> Loam Soil
5 -> Red Soil
6 -> Sandy Soil

Stage Mapping:
0 -> Flowering
1 -> Fruit/Grain/Bulb Formation
2 -> Germination
3 -> Harvest
4 -> Maturation
5 -> Pollination
6 -> Seedling Stage
7 -> Vegetative Growth / Root or Tuber Development


Full Model (All Features)

In [12]:
X = df[
    [
        "crop_ID_encoded",
        "soil_type_encoded",
        "stage_encoded",
        "MOI",
        "temp",
        "humidity"
    ]
]

y = df["result"]

print(X.shape)
print(y.shape)

(16411, 6)
(16411,)


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(13128, 6)
(3283, 6)


In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [15]:
from sklearn.metrics import accuracy_score, classification_report

pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

print("\nClassification Report:\n")
print(classification_report(y_test, pred))

Accuracy: 0.9872068230277186

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1813
           1       0.98      1.00      0.99      1246
           2       0.95      0.88      0.91       224

    accuracy                           0.99      3283
   macro avg       0.98      0.96      0.97      3283
weighted avg       0.99      0.99      0.99      3283



In [16]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

             Feature  Importance
3                MOI    0.361271
4               temp    0.283183
5           humidity    0.216885
2      stage_encoded    0.078946
0    crop_ID_encoded    0.033551
1  soil_type_encoded    0.026165


In [17]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)

print(cm)

[[1801    4    8]
 [   0 1244    2]
 [  12   16  196]]


Sensor-Only Model


In [20]:
# Features available from sensors

X = df[["MOI", "temp", "humidity"]]
y = df["result"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf_sensor = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_sensor.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [22]:
from sklearn.metrics import accuracy_score, classification_report

pred_sensor = rf_sensor.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_sensor))

print("\nClassification Report:\n")
print(classification_report(y_test, pred_sensor))

Accuracy: 0.8778556198598843

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.93      0.93      1813
           1       0.86      0.94      0.90      1246
           2       0.25      0.13      0.17       224

    accuracy                           0.88      3283
   macro avg       0.68      0.67      0.67      3283
weighted avg       0.86      0.88      0.87      3283



In [23]:
import joblib

joblib.dump(rf, "irrigation_model.pkl")

['irrigation_model.pkl']

In [24]:
joblib.dump(crop_encoder, "crop_encoder.pkl")
joblib.dump(soil_encoder, "soil_encoder.pkl")
joblib.dump(stage_encoder, "stage_encoder.pkl")

['stage_encoder.pkl']

Conclusion

1. Random Forest achieved 98.72% accuracy using all features.
2. MOI, Temperature, and Humidity were the most important features.
3. Sensor-only deployment achieved 87.79% accuracy.
4. A hybrid approach combining live sensor readings with user-provided crop and soil information provides the best performance.
5. The trained model is suitable for integration with a Raspberry Pi–based smart agriculture monitoring system.